# ✈️ Subsistema 02: Airspace Shield Autoencoder (`02-airspace-shield`)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/02_Airspace_ADSB_Autoencoder_Colab_GPU.ipynb)

* **Agências Líderes:** US Air Force (USAF), CBP Air and Marine Operations (AMO) & FAA
* **Anel de Defesa:** `RING_2_FEDERAL` (Espaço Aéreo da Fronteira Sul EUA-México e Caribe)
* **Arquitetura:** `FlightStateAutoencoder` (Vetores 4D: `[baro_altitude, velocity, vertical_rate, geo_altitude]`)
* **Objetivo:** Identificar aeronaves voando abaixo do radar (< 1.000 pés) sem plano de voo ou com transponder irregular vindas de pistas clandestinas na América do Sul e América Central.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FlightStateAutoencoder(nn.Module):
    def __init__(self, input_dim: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Linear(32, 12),
            nn.SiLU(),
            nn.Linear(12, 3),
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 12),
            nn.SiLU(),
            nn.Linear(12, 32),
            nn.SiLU(),
            nn.Linear(32, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        recon = self.decoder(self.encoder(x))
        mse = torch.mean((x - recon) ** 2, dim=-1, keepdim=True)
        return torch.sigmoid(mse * 12.0 - 2.0)

# Treinamento com vetores comerciais vs voos clandestinos de baixa altitude
x_commercial = torch.tensor(np.random.normal(0.7, 0.1, (2000, 4)).astype(np.float32), device=device)
model = FlightStateAutoencoder().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

history = []
for epoch in range(1, 41):
    model.train()
    opt.zero_grad()
    recon = model.decoder(model.encoder(x_commercial))
    loss = torch.mean((x_commercial - recon) ** 2)
    loss.backward()
    opt.step()
    history.append(loss.item())

plt.figure(figsize=(8, 3.5))
plt.plot(history, color="#f59e0b", linewidth=2, label="OpenSky ADS-B Reconstruction Loss")
plt.title("Treinamento FlightStateAutoencoder (USAF / CBP Air Shield)")
plt.xlabel("Época")
plt.ylabel("MSE Loss")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

model.eval()
torch.onnx.export(
    model,
    torch.randn(1, 4, device=device),
    "airspace_ae.onnx",
    input_names=["flight_vector"],
    output_names=["anomaly_probability"],
    dynamic_axes={"flight_vector": {0: "batch"}, "anomaly_probability": {0: "batch"}},
    opset_version=17,
)
print("✅ Exportado com sucesso: airspace_ae.onnx")
